# Download Raw Data

This notebook downloads all required raw input files into `data/raw` and merges OSM extracts into `data/raw/osm/merged/northern-germany.osm.pbf`.

Files handled:
- `population_grid.csv` (Destatis)
- `hvv_Rohdaten_GTFS_Fpl_26.ZIP` (HVV GTFS)
- `hamburg-260718.osm.pbf`
- `niedersachsen-260718.osm.pbf`
- `schleswig-holstein-260718.osm.pbf`
- merged `northern-germany.osm.pbf`

In [ ]:
from __future__ import annotations

import io
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from urllib.parse import urljoin

import requests

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from within the project directory.")
sys.path.insert(0, str(project_root / "src"))

import commute_analysis.config as config

In [ ]:
PROJECT_ROOT = config.PROJECT_ROOT
RAW_DIR = config.RAW_DATA_DIR
OSM_DIR = config.OSM_DATA_DIR
MERGED_DIR = config.MERGED_OSM_DIR

RAW_DIR.mkdir(parents=True, exist_ok=True)
OSM_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DIR)

Project root: /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis
Raw data dir: /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw


In [ ]:
def download_file(url: str, destination: Path, chunk_size: int = config.DOWNLOAD_CHUNK_SIZE_BYTES) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=config.DOWNLOAD_TIMEOUT_SECONDS) as response:
        response.raise_for_status()
        total = int(response.headers.get("Content-Length", "0"))
        written = 0
        with destination.open("wb") as file_handle:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if not chunk:
                    continue
                file_handle.write(chunk)
                written += len(chunk)
                if total:
                    percentage = (written / total) * 100
                    print(
                        f"\r{destination.name}: {percentage:5.1f}% "
                        f"({written:,}/{total:,} bytes)",
                        end="",
                    )
        if total:
            print()
    print(f"Saved {destination}")

## 1) Download population grid (Destatis)

Downloads the official ZIP and extracts `Zensus2022_Bevoelkerungszahl_100m-Gitter.csv` to `data/raw/population_grid.csv`.

In [ ]:
population_grid_path = config.POPULATION_GRID_PATH

response = requests.get(
    config.DESTATIS_POPULATION_ZIP_URL,
    timeout=config.DOWNLOAD_TIMEOUT_SECONDS,
)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    with archive.open(config.DESTATIS_POPULATION_CSV_FILENAME) as source_file:
        with population_grid_path.open("wb") as destination_file:
            shutil.copyfileobj(source_file, destination_file)

print("Saved", population_grid_path)

Saved /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw/population_grid.csv


## 2) Download HVV GTFS ZIP

Attempts to discover the direct file URL from the Hamburg transparency dataset page and saves it as `data/raw/hvv_Rohdaten_GTFS_Fpl_26.ZIP`.

In [ ]:
hvv_dataset_page = config.GTFS_DATASET_URL
hvv_target_path = config.RAW_DATA_DIR / config.GTFS_FILENAME

page = requests.get(
    hvv_dataset_page,
    timeout=config.DOWNLOAD_TIMEOUT_SECONDS,
)
page.raise_for_status()
html = page.text

hrefs = re.findall(r'href=["]([^"]+)["]', html, flags=re.IGNORECASE)
zip_candidates = []
for href in hrefs:
    lower = href.lower()
    if "gtfs" in lower and lower.endswith(".zip"):
        zip_candidates.append(urljoin(hvv_dataset_page, href))

if not zip_candidates:
    raise RuntimeError("Could not auto-discover GTFS ZIP URL from HVV dataset page.")

hvv_download_url = zip_candidates[0]
print("Using GTFS URL:", hvv_download_url)
download_file(hvv_download_url, hvv_target_path)

Using GTFS URL: https://daten.transparenz.hamburg.de/Dataport.HmbTG.ZS.Webservice.GetRessource100/GetRessource100.svc/f0073555-962c-4d55-870e-b94bb676ad9d/Upload__hvv_Rohdaten_GTFS_Fpl_20260408.ZIP
Saved /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw/hvv_Rohdaten_GTFS_Fpl_26.ZIP


## 3) Download OSM PBF extracts

Downloads the three required Geofabrik extracts into `data/raw/osm/`.

In [ ]:
for filename, url in config.OSM_DOWNLOAD_URLS.items():
    download_file(url, OSM_DIR / filename)

hamburg-260718.osm.pbf: 100.0% (53,621,741/53,621,741 bytes)
Saved /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw/osm/hamburg-260718.osm.pbf
niedersachsen-260718.osm.pbf: 100.0% (503,816,224/503,816,224 bytes)
Saved /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw/osm/niedersachsen-260718.osm.pbf
schleswig-holstein-260718.osm.pbf: 100.0% (157,738,432/157,738,432 bytes)
Saved /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw/osm/schleswig-holstein-260718.osm.pbf


## 4) Merge OSM PBF files

Requires `osmium` CLI. On Ubuntu/Debian:

```bash
sudo apt install -y osmium-tool
```

Merges the three regional extracts into `data/raw/osm/merged/northern-germany.osm.pbf`.

In [ ]:
osmium_path = shutil.which(config.OSMIUM_COMMAND)
if osmium_path is None:
    raise RuntimeError(
        f"{config.OSMIUM_COMMAND} CLI is not installed. "
        "Install it using your system package manager."
    )

merged_output = config.MERGED_OSM_PATH
inputs = [OSM_DIR / filename for filename in config.OSM_PBF_FILENAMES]

missing = [str(path) for path in inputs if not path.exists()]
if missing:
    raise FileNotFoundError("Missing OSM input files: " + ", ".join(missing))

command = [
    osmium_path,
    "merge",
    *(str(path) for path in inputs),
    "-o",
    str(merged_output),
]
if config.OVERWRITE_MERGED_OSM:
    command.append("--overwrite")
print("Running:", " ".join(command))
subprocess.run(command, check=True)
print("Saved", merged_output)

Running: /usr/bin/osmium merge /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw/osm/hamburg-260718.osm.pbf /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw/osm/niedersachsen-260718.osm.pbf /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw/osm/schleswig-holstein-260718.osm.pbf -o /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw/osm/merged/northern-germany.osm.pbf --overwrite
Saved /home/moritz/pythonproj/deticket/deutschlandticket_commute_analysis/data/raw/osm/merged/northern-germany.osm.pbf
